In [86]:
from pathlib import Path
import pandas as pd

# Folder where mode2 trajectory pickles are expected
# TRAJ_DIR = Path("exports/dft_2equiv")
TRAJ_DIR = Path("exports/dft_literature-conditions")

# You can set a specific file name here, or keep None to auto-pick first match
TRAJ_FILE = "im3-j26_mode2_trajectories.pkl.gz"  # e.g., "im1-w1m_mode2_trajectories.pkl.gz"

# Products to compare at final time
PRODUCT_A = "map"
PRODUCT_B = "bis"

In [87]:
def load_trajectory_pickle(traj_dir: Path, traj_file: str | None = None) -> pd.DataFrame:
    if not traj_dir.exists():
        raise FileNotFoundError(f"Trajectory directory not found: {traj_dir}")

    if traj_file is not None:
        target = traj_dir / traj_file
        if not target.exists():
            raise FileNotFoundError(f"Trajectory file not found: {target}")
        return pd.read_pickle(target, compression="gzip")

    candidates = sorted(traj_dir.glob("*_mode2_trajectories.pkl.gz"))
    if not candidates:
        raise FileNotFoundError(
            f"No trajectory pickle files found in {traj_dir}. "
            "Run Mode 2 export first."
        )
    print(f"Using trajectory file: {candidates[0]}")
    return pd.read_pickle(candidates[0], compression="gzip")


def classify_major_product_last_point(
    traj_df: pd.DataFrame,
    product_a: str = "map",
    product_b: str = "bis",
) -> pd.DataFrame:
    required_cols = {"sample", "time", product_a, product_b}
    missing = required_cols - set(traj_df.columns)
    if missing:
        raise KeyError(f"Missing required columns in trajectory data: {sorted(missing)}")

    # Last time point per sample
    last_points = (
        traj_df.sort_values(["sample", "time"])
        .groupby("sample", as_index=False)
        .tail(1)
        .copy()
    )

    last_points["major_product"] = last_points.apply(
        lambda r: product_a if r[product_a] > r[product_b]
        else (product_b if r[product_b] > r[product_a] else "tie"),
        axis=1,
    )

    summary = (
        last_points["major_product"]
        .value_counts(dropna=False)
        .rename_axis("major_product")
        .reset_index(name="n_samples")
    )

    print("Major-product counts at final time point:")
    display(summary)

    return last_points[["sample", "time", "bispyr", product_a, product_b, "major_product"]]


traj_df = load_trajectory_pickle(TRAJ_DIR, TRAJ_FILE)
print(f"Loaded rows: {len(traj_df)}")
display(traj_df.head())

sample_major_product = classify_major_product_last_point(
    traj_df,
    product_a=PRODUCT_A,
    product_b=PRODUCT_B,
)

display(sample_major_product.head())
display(sample_major_product.tail())

display(sample_major_product.describe().round(2))

Loaded rows: 1000000


,sample,reaction,time,bispyr,roh,me2pyr,map,bis
0,0,im3-j26,0.000000,1.0,1.0,0.000000e+00,0.000000e+00,0.000000e+00
1,0,im3-j26,0.000004,1.0,1.0,3.915068e-11,3.915068e-11,5.353733e-32
2,0,im3-j26,0.000029,1.0,1.0,3.132055e-10,3.132055e-10,4.282986e-31
3,0,im3-j26,0.000097,1.0,1.0,1.057068e-09,1.057068e-09,1.445508e-30
4,0,im3-j26,0.000231,1.0,1.0,2.505644e-09,2.505644e-09,9.130135e-30


Major-product counts at final time point:


,major_product,n_samples
0,map,1000


,sample,time,bispyr,map,bis,major_product
999,0,3600.0,0.962433,0.037567,1.041404e-15,map
1999,1,3600.0,0.989768,0.010232,3.467273e-16,map
2999,2,3600.0,0.024038,0.975962,1.806765e-15,map
3999,3,3600.0,0.958112,0.041888,7.104233e-16,map
4999,4,3600.0,0.940559,0.059441,8.843331e-16,map


,sample,time,bispyr,map,bis,major_product
995999,995,3600.0,0.963080,0.036920,1.859835e-15,map
996999,996,3600.0,0.934257,0.065743,8.505616e-16,map
997999,997,3600.0,0.985297,0.014703,8.403577e-16,map
998999,998,3600.0,0.940126,0.059874,3.339720e-16,map
999999,999,3600.0,0.992963,0.007037,4.122600e-15,map


,sample,time,bispyr,map,bis
count,1000.00,1000.0,1000.00,1000.00,1000.0
mean,499.50,3600.0,0.87,0.13,0.0
std,288.82,0.0,0.21,0.21,0.0
min,0.00,3600.0,0.00,0.00,0.0
25%,249.75,3600.0,0.88,0.03,0.0
50%,499.50,3600.0,0.94,0.06,0.0
75%,749.25,3600.0,0.97,0.12,0.0
max,999.00,3600.0,1.00,1.00,0.0
